<a href="https://colab.research.google.com/github/mmallare/ECGR4106/blob/main/ecgr4106_homework5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
import time
import pandas as pd

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hyperparameters (shared across all models)
image_size = 32
num_classes = 100
num_epochs = 10
batch_size = 64
learning_rate = 0.001

# Data preparation
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# CIFAR-100 dataset
train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True,
                                               download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False,
                                              download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Patch embedding layer
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels=3, embed_dim=256):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                            kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)         # [B, embed_dim, H', W']
        x = x.flatten(2)         # [B, embed_dim, num_patches]
        x = x.transpose(1, 2)    # [B, num_patches, embed_dim]
        return x

# Transformer Encoder
class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.layer_norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x2 = self.layer_norm1(x)
        attention_output, _ = self.attention(x2, x2, x2)
        x = x + attention_output
        x2 = self.layer_norm2(x)
        mlp_output = self.mlp(x2)
        x = x + mlp_output
        return x

# Vision Transformer
class VisionTransformer(nn.Module):
    def __init__(self, image_size, patch_size, num_classes, embed_dim,
                 num_heads, num_layers, mlp_dim, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size, patch_size, 3, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        num_patches = (image_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        self.transformer = nn.ModuleList(
            [TransformerEncoder(embed_dim, num_heads, mlp_dim, dropout)
             for _ in range(num_layers)]
        )

        self.layer_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)

        for transformer in self.transformer:
            x = transformer(x)

        x = self.layer_norm(x)
        cls_token_final = x[:, 0]
        x = self.head(cls_token_final)
        return x

In [ ]:
def build_resnet18(num_classes=100, pretrained=True):
    if pretrained:
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    else:
        resnet = models.resnet18(weights=None)
    resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
    return resnet

In [ ]:
!pip install thop -q
from thop import profile

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_flops(model):
    model.eval()
    dummy_input = torch.randn(1, 3, image_size, image_size).to(device)
    flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
    return flops

In [ ]:
def train(model, optimizer, name):
    criterion = nn.CrossEntropyLoss()
    epoch_times = []
    best_acc = 0.0
    best_epoch = 0

    for epoch in range(num_epochs):
        model.train()
        start_time = time.time()
        for i, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (i+1) % 200 == 0:
                print(f'[{name}] Epoch [{epoch+1}/{num_epochs}], Step [{i+1}], Loss: {loss.item():.4f}')

        epoch_time = time.time() - start_time
        epoch_times.append(epoch_time)

        # Evaluate on test set after every epoch
        epoch_acc = test(model, name)

        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_epoch = epoch + 1

        print(f'[{name}] Epoch [{epoch+1}/{num_epochs}] completed in {epoch_time:.1f}s | Test Acc: {epoch_acc:.2f}%')

    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    final_acc = epoch_acc  # accuracy from the last epoch
    return avg_epoch_time, final_acc, best_acc, best_epoch


def test(model, name):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        return accuracy

In [ ]:
vit_configs = {
    "ViT-P4-E256-D4-H4": dict(patch_size=4, embed_dim=256, num_layers=4, num_heads=4, mlp_dim=1024),
    "ViT-P4-E256-D8-H8": dict(patch_size=4, embed_dim=256, num_layers=8, num_heads=8, mlp_dim=1024),
    "ViT-P8-E512-D4-H8": dict(patch_size=8, embed_dim=512, num_layers=4, num_heads=8, mlp_dim=2048),
    "ViT-P8-E512-D8-H8": dict(patch_size=8, embed_dim=512, num_layers=8, num_heads=8, mlp_dim=2048),
}

results = []

# ViT configurations
for name, cfg in vit_configs.items():
    print(f"\n===== {name} =====")
    model = VisionTransformer(
        image_size=image_size,
        patch_size=cfg["patch_size"],
        num_classes=num_classes,
        embed_dim=cfg["embed_dim"],
        num_heads=cfg["num_heads"],
        num_layers=cfg["num_layers"],
        mlp_dim=cfg["mlp_dim"]
    ).to(device)

    params = count_params(model)
    flops = count_flops(model)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    print("Training started...")
    avg_epoch_time, final_acc, best_acc, best_epoch = train(model, optimizer, name)

    results.append({
        "Model": name,
        "Params (M)": round(params / 1e6, 2),
        "FLOPs (G)": round(flops / 1e9, 3),
        "Avg Epoch Time (s)": round(avg_epoch_time, 1),
        "Final Test Acc (%)": round(final_acc, 2),
        "Best Test Acc (%)": round(best_acc, 2),
        "Best Epoch": best_epoch,
    })

# ResNet-18 baseline
print("\n===== ResNet-18 =====")
resnet = build_resnet18(num_classes=num_classes, pretrained=True).to(device)
params = count_params(resnet)
flops = count_flops(resnet)
optimizer = torch.optim.Adam(resnet.parameters(), lr=learning_rate)

print("Training started...")
avg_epoch_time, final_acc, best_acc, best_epoch = train(resnet, optimizer, "ResNet-18")

results.append({
    "Model": "ResNet-18",
    "Params (M)": round(params / 1e6, 2),
    "FLOPs (G)": round(flops / 1e9, 3),
    "Avg Epoch Time (s)": round(avg_epoch_time, 1),
    "Final Test Acc (%)": round(final_acc, 2),
    "Best Test Acc (%)": round(best_acc, 2),
    "Best Epoch": best_epoch,
})

# Summary table
df = pd.DataFrame(results)
print("\n\n===== SUMMARY TABLE =====")
print(df.to_string(index=False))
df.to_csv("results.csv", index=False)


===== ViT-P4-E256-D4-H4 =====
Training started...
[ViT-P4-E256-D4-H4] Epoch [1/10], Step [200], Loss: 3.8718
[ViT-P4-E256-D4-H4] Epoch [1/10], Step [400], Loss: 3.8683
[ViT-P4-E256-D4-H4] Epoch [1/10], Step [600], Loss: 3.5161
[ViT-P4-E256-D4-H4] Epoch [1/10] completed in 21.6s | Test Acc: 13.39%
[ViT-P4-E256-D4-H4] Epoch [2/10], Step [200], Loss: 3.4879
[ViT-P4-E256-D4-H4] Epoch [2/10], Step [400], Loss: 3.2184
[ViT-P4-E256-D4-H4] Epoch [2/10], Step [600], Loss: 3.3506
[ViT-P4-E256-D4-H4] Epoch [2/10] completed in 21.6s | Test Acc: 21.27%
[ViT-P4-E256-D4-H4] Epoch [3/10], Step [200], Loss: 3.1697
[ViT-P4-E256-D4-H4] Epoch [3/10], Step [400], Loss: 3.2106
[ViT-P4-E256-D4-H4] Epoch [3/10], Step [600], Loss: 3.0127
[ViT-P4-E256-D4-H4] Epoch [3/10] completed in 21.3s | Test Acc: 24.51%
[ViT-P4-E256-D4-H4] Epoch [4/10], Step [200], Loss: 3.1359
[ViT-P4-E256-D4-H4] Epoch [4/10], Step [400], Loss: 3.0760
[ViT-P4-E256-D4-H4] Epoch [4/10], Step [600], Loss: 2.8272
[ViT-P4-E256-D4-H4] Epoch [4

Problem 2

In [ ]:
!pip install transformers -q

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from transformers import SwinForImageClassification
import time
import pandas as pd

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hyperparameters
image_size = 224
num_classes = 100
num_epochs = 5
batch_size = 32

lr_finetune = 2e-5
lr_scratch = 0.001

# Preprocessing
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True,
                                               download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False,
                                              download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def build_pretrained_swin(model_name, num_classes=100):
    model = SwinForImageClassification.from_pretrained(
        model_name,
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    )

    # Freeze the backbone
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

    return model

swin_tiny = build_pretrained_swin("microsoft/swin-tiny-patch4-window7-224", num_classes).to(device)
swin_small = build_pretrained_swin("microsoft/swin-small-patch4-window7-224", num_classes).to(device)

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin:   0%|          | 0.00/199M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/425 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-small-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows

def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x

# --- Patch embedding ---
class SwinPatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size=4, in_channels=3, embed_dim=96):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.grid_size = image_size // patch_size

    def forward(self, x):
        x = self.proj(x)
        H, W = x.shape[2], x.shape[3]
        x = x.flatten(2).transpose(1, 2)
        return x, H, W
-
class WindowAttention(nn.Module):
    def __init__(self, embed_dim, window_size, num_heads):
        super().__init__()
        self.window_size = window_size
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        out = self.proj(out)
        return out

# Swin Transformer block
class SwinBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, window_size=7, shift_size=0, mlp_ratio=4):
        super().__init__()
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = WindowAttention(embed_dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(embed_dim * mlp_ratio, embed_dim)
        )

    def make_mask(self, H, W, device):
        img_mask = torch.zeros((1, H, W, 1), device=device)
        h_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
        w_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1

        mask_windows = window_partition(img_mask, self.window_size).view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
        return attn_mask

    def forward(self, x, H, W):
        B, N, C = x.shape
        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
            mask = self.make_mask(H, W, x.device)
        else:
            mask = None

        x_windows = window_partition(x, self.window_size).view(-1, self.window_size * self.window_size, C)
        attn_out = self.attn(x_windows, mask=mask)
        attn_out = attn_out.view(-1, self.window_size, self.window_size, C)
        x = window_reverse(attn_out, self.window_size, H, W)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))

        x = x.view(B, H * W, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x

# Patch merging
class PatchMerging(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.reduction = nn.Linear(4 * embed_dim, 2 * embed_dim)
        self.norm = nn.LayerNorm(4 * embed_dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.view(B, H, W, C)
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], dim=-1)
        x = x.view(B, -1, 4 * C)
        x = self.norm(x)
        x = self.reduction(x)
        return x, H // 2, W // 2

# Full Swin Transformer
class SwinTransformerScratch(nn.Module):
    def __init__(self, image_size=224, patch_size=4, in_channels=3, num_classes=100,
                 embed_dim=96, depths=(2, 2), num_heads=(3, 6), window_size=7):
        super().__init__()
        self.patch_embed = SwinPatchEmbedding(image_size, patch_size, in_channels, embed_dim)

        self.stages = nn.ModuleList()
        self.merges = nn.ModuleList()
        dim = embed_dim
        for stage_idx, depth in enumerate(depths):
            blocks = nn.ModuleList([
                SwinBlock(dim, num_heads[stage_idx], window_size,
                          shift_size=0 if (i % 2 == 0) else window_size // 2)
                for i in range(depth)
            ])
            self.stages.append(blocks)
            if stage_idx < len(depths) - 1:
                self.merges.append(PatchMerging(dim))
                dim *= 2
            else:
                self.merges.append(None)

        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        x, H, W = self.patch_embed(x)

        for stage_idx, blocks in enumerate(self.stages):
            for block in blocks:
                x = block(x, H, W)
            if self.merges[stage_idx] is not None:
                x, H, W = self.merges[stage_idx](x, H, W)

        x = self.norm(x)
        x = x.mean(dim=1)
        x = self.head(x)
        return x

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_total_params(model):
    return sum(p.numel() for p in model.parameters())

def train(model, optimizer, name, get_logits=lambda out: out):
    criterion = nn.CrossEntropyLoss()
    epoch_times = []
    best_acc = 0.0
    best_epoch = 0

    for epoch in range(num_epochs):
        model.train()
        start_time = time.time()
        for i, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            logits = get_logits(outputs)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (i+1) % 200 == 0:
                print(f'[{name}] Epoch [{epoch+1}/{num_epochs}], Step [{i+1}], Loss: {loss.item():.4f}')

        epoch_time = time.time() - start_time
        epoch_times.append(epoch_time)

        epoch_acc = test(model, name, get_logits)
        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_epoch = epoch + 1

        print(f'[{name}] Epoch [{epoch+1}/{num_epochs}] completed in {epoch_time:.1f}s | Test Acc: {epoch_acc:.2f}%')

    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    final_acc = epoch_acc
    return avg_epoch_time, final_acc, best_acc, best_epoch


def test(model, name, get_logits=lambda out: out):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            logits = get_logits(outputs)
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        return accuracy

In [ ]:
results = []

# Swin-Tiny
print("\n===== Swin-Tiny (pretrained) =====")
trainable_params = count_params(swin_tiny)
total_params = count_total_params(swin_tiny)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, swin_tiny.parameters()), lr=lr_finetune)

avg_epoch_time, final_acc, best_acc, best_epoch = train(
    swin_tiny, optimizer, "Swin-Tiny", get_logits=lambda out: out.logits)

results.append({
    "Model": "Swin-Tiny (pretrained)",
    "Total Params (M)": round(total_params / 1e6, 2),
    "Trainable Params (M)": round(trainable_params / 1e6, 2),
    "Avg Epoch Time (s)": round(avg_epoch_time, 1),
    "Final Test Acc (%)": round(final_acc, 2),
    "Best Test Acc (%)": round(best_acc, 2),
    "Best Epoch": best_epoch,
})

# Swin-Small
print("\n===== Swin-Small (pretrained) =====")
trainable_params = count_params(swin_small)
total_params = count_total_params(swin_small)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, swin_small.parameters()), lr=lr_finetune)

avg_epoch_time, final_acc, best_acc, best_epoch = train(
    swin_small, optimizer, "Swin-Small", get_logits=lambda out: out.logits)

results.append({
    "Model": "Swin-Small (pretrained)",
    "Total Params (M)": round(total_params / 1e6, 2),
    "Trainable Params (M)": round(trainable_params / 1e6, 2),
    "Avg Epoch Time (s)": round(avg_epoch_time, 1),
    "Final Test Acc (%)": round(final_acc, 2),
    "Best Test Acc (%)": round(best_acc, 2),
    "Best Epoch": best_epoch,
})

# Swin Transformer from scratch
print("\n===== Swin (scratch) =====")
swin_scratch = SwinTransformerScratch(
    image_size=image_size, num_classes=num_classes,
    embed_dim=96, depths=(2, 2), num_heads=(3, 6), window_size=7
).to(device)

trainable_params = count_params(swin_scratch)
total_params = count_total_params(swin_scratch)
optimizer = torch.optim.Adam(swin_scratch.parameters(), lr=lr_scratch)

avg_epoch_time, final_acc, best_acc, best_epoch = train(swin_scratch, optimizer, "Swin-Scratch")

results.append({
    "Model": "Swin (scratch)",
    "Total Params (M)": round(total_params / 1e6, 2),
    "Trainable Params (M)": round(trainable_params / 1e6, 2),
    "Avg Epoch Time (s)": round(avg_epoch_time, 1),
    "Final Test Acc (%)": round(final_acc, 2),
    "Best Test Acc (%)": round(best_acc, 2),
    "Best Epoch": best_epoch,
})

# Summary table
df = pd.DataFrame(results)
print("\n\n===== SUMMARY TABLE (Problem 2) =====")
print(df.to_string(index=False))
df.to_csv("results_problem2.csv", index=False)


===== Swin-Tiny (pretrained) =====
[Swin-Tiny] Epoch [1/5], Step [200], Loss: 4.4185
[Swin-Tiny] Epoch [1/5], Step [400], Loss: 4.3598
[Swin-Tiny] Epoch [1/5], Step [600], Loss: 4.3920
[Swin-Tiny] Epoch [1/5], Step [800], Loss: 4.2187
[Swin-Tiny] Epoch [1/5], Step [1000], Loss: 4.1105
[Swin-Tiny] Epoch [1/5], Step [1200], Loss: 3.9777
[Swin-Tiny] Epoch [1/5], Step [1400], Loss: 3.9613
[Swin-Tiny] Epoch [1/5] completed in 87.0s | Test Acc: 43.37%
[Swin-Tiny] Epoch [2/5], Step [200], Loss: 3.9449
[Swin-Tiny] Epoch [2/5], Step [400], Loss: 3.6454
[Swin-Tiny] Epoch [2/5], Step [600], Loss: 3.6799
[Swin-Tiny] Epoch [2/5], Step [800], Loss: 3.4050
[Swin-Tiny] Epoch [2/5], Step [1000], Loss: 3.4074
[Swin-Tiny] Epoch [2/5], Step [1200], Loss: 3.3370
[Swin-Tiny] Epoch [2/5], Step [1400], Loss: 3.4657
[Swin-Tiny] Epoch [2/5] completed in 85.8s | Test Acc: 54.60%
[Swin-Tiny] Epoch [3/5], Step [200], Loss: 3.0541
[Swin-Tiny] Epoch [3/5], Step [400], Loss: 3.0426
[Swin-Tiny] Epoch [3/5], Step [600